In [18]:
import pandas as pd
from pathlib import Path
import cv2
from pyzbar.pyzbar import decode
import numpy as np

In [34]:
dfs_path = Path('../data/dfs/')
images_path = Path('../data/images/')
rejected_images_path = images_path / Path('rejected_images/')
rejected_images_path.mkdir(parents=True, exist_ok=True)

In [35]:
df_images = pd.read_csv(dfs_path / 'images.csv', index_col=0)
df_images.head()

,image_path,bbox,page,doc_id
0,doc0_page0_3094.jpeg,"(0.0, 0.0, 467.760009765625, 595.2000122070312)",0,0
1,doc0_page1_504.png,"(145.8669891357422, 465.240966796875, 291.1790...",1,0
2,doc0_page9_640.png,"(36.85051727294922, 185.30657958984375, 375.48...",9,0
3,doc0_page9_642.png,"(36.85051727294922, 185.30657958984375, 375.48...",9,0
4,doc0_page11_660.png,"(22.677440643310547, 417.9521789550781, 89.821...",11,0


In [36]:
def is_qrcode(gray_image):
    detected_objects = decode(gray_image)
    if detected_objects:
        return True

    inverted_image = cv2.bitwise_not(gray_image)
    detected_objects = decode(inverted_image)

    return len(detected_objects) > 0


In [37]:
def is_gradient(gray_image, threshold=100):

    h, w = gray_image.shape

    if h < 100 and w < 100:
        return True

    margin_h = int(h * 0.1)
    margin_w = int(w * 0.1)

    if h > 2 * margin_h and w > 2 * margin_w:
        cropped_image = gray_image[margin_h:h - margin_h, margin_w:w - margin_w]
    else:
        return True

    laplacian_var = cv2.Laplacian(cropped_image, cv2.CV_64F).var()
    return laplacian_var < threshold

In [38]:
qrcode_results = []
gradient_results = []
corrupted_results = []

for index, image_row in df_images.iterrows():
    image_path = images_path / image_row['image_path']
    image = cv2.imread(image_path)

    if image is None:
        qrcode_results.append(False)
        gradient_results.append(False)
        corrupted_results.append(True)
        continue

    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    qrcode_results.append(is_qrcode(gray_image))
    gradient_results.append(is_gradient(gray_image))

    if qrcode_results[index] or gradient_results[index] or corrupted_results[index]:
        image_path.rename(rejected_images_path / image_row['image_path'])

df_images['is_qrcode'] = qrcode_results
df_images['is_gradient'] = gradient_results

image_path


[ WARN:0@3170.048] global loadsave.cpp:275 findDecoder imread_('../data/images/doc11_page0_1657.png'): can't open/read file: check file path/integrity


ValueError: Length of values (6485) does not match length of index (6486)